We’ll cover:

1️⃣ Connecting to EC2
2️⃣ Installing Node + npm
3️⃣ Setting up the project
4️⃣ Running the app (with `nohup`)
5️⃣ AWS network/security setup (so it’s reachable via IP)

---

## 🚀 STEP-BY-STEP GUIDE

### 1️⃣ Connect to your EC2 instance

From your local terminal:

```bash
ssh -i /path/to/your-key.pem ubuntu@<your-ec2-public-ip>
```

Example:

```bash
ssh -i ~/Downloads/mykey.pem ubuntu@13.232.56.101
```

---

### 2️⃣ Update packages

```bash
sudo apt update && sudo apt upgrade -y
```

---

### 3️⃣ Install Node.js and npm

For Vite projects you need Node ≥ 18 or 20.

```bash
# Option 1: use NodeSource (recommended)
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt install -y nodejs

# verify
node -v
npm -v
```

---

### 4️⃣ Clone or upload your React + Vite project

Either:

#### ✅ Clone from GitHub:

```bash
cd ~
git clone https://github.com/yourusername/yourproject.git
cd yourproject
```

or

#### ✅ Upload (via SCP or FileZilla):

```bash
scp -i /path/to/your-key.pem -r ./yourproject ubuntu@<your-ec2-public-ip>:/home/ubuntu/
```

---

### 5️⃣ Install dependencies

```bash
cd yourproject
npm install
```

---

### 6️⃣ Edit the Vite config (important!)

Open `vite.config.js` and ensure you expose all network interfaces:

```js
export default {
  server: {
    host: '0.0.0.0',
    port: 5173, // default vite port
  },
}
```

---

### 7️⃣ Run the development server (test)

```bash
npm run dev
```

You should see:

```
Local: http://localhost:5173/
Network: http://<your-ec2-private-ip>:5173/
```

✅ If you see that, it’s working locally.

---

### 8️⃣ Allow inbound traffic in AWS

1. Go to **EC2 → Instances → Your Instance → Security → Security groups**
2. Edit **Inbound rules**
3. Add rule:

   * **Type:** Custom TCP
   * **Port range:** 5173
   * **Source:** 0.0.0.0/0 (and optionally ::/0 for IPv6)

Save.

Now anyone can reach `http://<your-ec2-public-ip>:5173`

---

### 9️⃣ Run it in background (nohup)

Stop the current process (`Ctrl +C`) and run:

```bash
nohup npm run dev -- --host 0.0.0.0 > output.log 2>&1 &
```

✅ This will:

* run your Vite dev server in background
* write logs to `output.log`
* survive terminal logout

Check running processes:

```bash
ps aux | grep vite
```

Kill if needed:

```bash
kill -9 <PID>
```

---

### 10️⃣ Access the frontend

Open your browser:

```
http://<your-ec2-public-ip>:5173
```

Example:

```
http://13.232.56.101:5173
```

---

## 🧩 (OPTIONAL) – Build + Serve Static Files

If you prefer to serve a built version (recommended for production):

```bash
npm run build
sudo npm install -g serve
nohup serve -s dist -l 3000 > serve.log 2>&1 &
```

Then update AWS inbound rule:

* Port 3000 → 0.0.0.0/0

Access:

```
http://<your-ec2-public-ip>:3000
```

---

## ✅ Quick Summary

| Step | Command / Action                              | Purpose               |
| ---- | --------------------------------------------- | --------------------- |
| 1    | `ssh -i key.pem ubuntu@ip`                    | connect to EC2        |
| 2    | `sudo apt update && sudo apt upgrade -y`      | update packages       |
| 3    | install Node 20                               | node + npm            |
| 4    | `git clone …` / `scp`                         | get project           |
| 5    | `npm install`                                 | install deps          |
| 6    | edit `vite.config.js` → `host:'0.0.0.0'`      | allow external access |
| 7    | `npm run dev`                                 | test run              |
| 8    | add inbound rule (port 5173/3000 → 0.0.0.0/0) | open port             |
| 9    | `nohup npm run dev -- --host 0.0.0.0 &`       | keep running          |
| 10   | open `http://<EC2-IP>:5173`                   | view app              |

---



# **Nginx reverse-proxy setup** (so your React app runs on port 80 without typing `:5173`)? It’s the usual next step for production hosting.

Perfect 👌 — now let’s continue from your **working Vite/React app** on EC2 and make it run on **port 80 (default HTTP)**, so you can access it just via:
👉 `http://<your-ec2-public-ip>`
without typing `:5173` or `:3000`.

We’ll do this using **Nginx as a reverse proxy**.

---

## 🌐 Full Step-by-Step NGINX Reverse Proxy Setup (Ubuntu EC2)

### 🔹 1️⃣ Install Nginx

```bash
sudo apt update
sudo apt install nginx -y
```

Verify installation:

```bash
sudo systemctl status nginx
```

If not running:

```bash
sudo systemctl start nginx
sudo systemctl enable nginx
```

Now check in browser:
👉 `http://<your-ec2-public-ip>`
You should see the **default Nginx welcome page**.

---

### 🔹 2️⃣ Open Port 80 in AWS Security Group

Go to:

* **EC2 → Instances → Your instance → Security → Security groups**
* Edit **Inbound rules**
  Add:

  * **Type:** HTTP
  * **Port:** 80
  * **Source:** `0.0.0.0/0`
    (and optionally `::/0` for IPv6)

Save.

---

### 🔹 3️⃣ Run your Vite/React app in background (on internal port)

If you already built your app, serve it with:

```bash
npm run build
sudo npm install -g serve
nohup serve -s dist -l 5173 > frontend.log 2>&1 &
```

*(or whatever internal port you want; 5173 is fine)*

Check it’s running:

```bash
curl http://localhost:5173
```

If you see HTML output → ✅ it’s serving fine.

---

### 🔹 4️⃣ Configure Nginx reverse proxy

Now tell Nginx to forward traffic from **port 80 → port 5173**.

Create or edit config file:

```bash
sudo nano /etc/nginx/sites-available/reactapp
```

Add this content:

```nginx
server {
    listen 80;
    server_name _;

    location / {
        proxy_pass http://127.0.0.1:5173;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection 'upgrade';
        proxy_set_header Host $host;
        proxy_cache_bypass $http_upgrade;
    }
}
```

Save (Ctrl + O, Enter, Ctrl + X).

---

### 🔹 5️⃣ Enable the site and reload Nginx

```bash
sudo ln -s /etc/nginx/sites-available/reactapp /etc/nginx/sites-enabled/
sudo nginx -t  # check syntax
sudo systemctl reload nginx
```

---

### 🔹 6️⃣ Test it

Now open your browser:

```
http://<your-ec2-public-ip>
```

✅ You should see your React/Vite app running **without `:5173`**.

---

### 🔹 7️⃣ Optional: Remove default Nginx page

If you want to disable the default page:

```bash
sudo rm /etc/nginx/sites-enabled/default
sudo systemctl reload nginx
```

---

### 🔹 8️⃣ (Optional) Use custom domain with SSL (bonus)

If you later map a domain (e.g., `myapp.com`) to your EC2 IP, you can easily add HTTPS:

```bash
sudo apt install certbot python3-certbot-nginx -y
sudo certbot --nginx -d myapp.com -d www.myapp.com
```

Certbot will automatically generate and configure SSL certificates.

---

## ✅ Summary

| Step | Command / Action                               | Description           |
| ---- | ---------------------------------------------- | --------------------- |
| 1    | `sudo apt install nginx -y`                    | install nginx         |
| 2    | AWS SG inbound rule → port 80                  | open HTTP access      |
| 3    | `nohup serve -s dist -l 5173 &`                | run frontend          |
| 4    | edit `/etc/nginx/sites-available/reactapp`     | proxy config          |
| 5    | `sudo nginx -t && sudo systemctl reload nginx` | apply config          |
| 6    | visit `http://<EC2-IP>`                        | app runs without port |
| 7    | `sudo rm /etc/nginx/sites-enabled/default`     | optional cleanup      |

---
